# GeoTransformer for Faces — Pipeline Inspection

This notebook walks through every conceptual step the model takes to register two face point clouds,
extracting and visualising all intermediate results.

---

## Conceptual Pipeline

| # | Step | Module |
|---|------|---------|
| 1 | **Data preparation** — collate ref+src into multi-scale KNN graphs | `registration_collate_fn_stack_mode` |
| 2 | **Pass-1 Backbone** — KPConvFPN encodes ref+src at 4 scales (encoder) then decodes back to 2 fine scales (FPN decoder) | `KPConvFPN` |
| 3 | **Shape morphing — coefficient regression** — coarse src features are fed into a cross-attention regressor that predicts 32×100 PCA coefficients | `CrossAttentionRegressor` |
| 4 | **Generate morphed reference** — decode coefficients through a pre-trained PCA basis to produce a personalised reference face (morphed_full) | `generate_reference_geometry` |
| 5 | **Pass-2 Backbone** — re-run KPConvFPN on morphed ref + src (detached morph prevents gradient leakage into rigid-alignment losses) | `KPConvFPN` |
| 6 | **Point-to-node partitioning** — assign fine-level points to coarse superpoints; each superpoint owns a patch of ≤64 fine neighbours | `point_to_node_partition` |
| 7 | **Geometric Structure Embedding** — encode pairwise distances and triplet-wise angles among superpoints as sinusoidal position biases | `GeometricStructureEmbedding` |
| 8 | **Geometric Transformer** — alternating self/cross attention with geometry-conditioned RPE refines coarse features to be globally context-aware and cross-cloud aware | `GeometricTransformer` |
| 9 | **Coarse matching** — dual-normalised feature similarity selects the top-256 superpoint correspondences | `SuperPointMatching` |
| 10 | **Fine matching prep** — for every coarse match gather the fine-level kNN point patches from both clouds | index gather |
| 11 | **Optimal Transport (Sinkhorn)** — learnable log-Sinkhorn produces a soft assignment matrix between fine points within each patch pair | `LearnableLogOptimalTransport` |
| 12 | **Local-Global Registration** — top-k fine correspondences per patch → weighted Procrustes → per-patch local transforms → aggregate + iterative refinement → final rigid transform | `LocalGlobalRegistration` |
| 13 | **Result** — estimated 4×4 rigid transform aligns src onto ref | — |

---

## 0. Setup

In [ ]:
import sys, os

# Point to the experiment folder so we can import config/model/backbone
#EXP_DIR = os.path.abspath('../experiments/geotransformer.faces.stage4.gse.k3.max.oacl.stage2.sinkhorn')
#sys.path.insert(0, EXP_DIR)

# Root of the repo (contains the `geotransformer` package)
#REPO_DIR = os.path.abspath('..')
#sys.path.insert(0, REPO_DIR)

import torch
import numpy as np

import torch.nn.functional as F
import matplotlib.pyplot as plt
import open3d as o3d
from mpl_toolkits.mplot3d import Axes3D  # noqa

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
from geotransformer.utils.data import registration_collate_fn_stack_mode
from geotransformer.utils.torch import to_cuda, release_cuda
from geotransformer.utils.open3d import make_open3d_point_cloud, get_color
from geotransformer.utils.registration import compute_registration_error
from geotransformer.modules.ops import point_to_node_partition, index_select
from geotransformer.modules.registration import get_node_correspondences

from config_dowsampled import make_cfg
from model import create_model, GeoTransformer
from backbone import KPConvFPN

In [ ]:
REPO_DIR = '../../'

### Load model weights

In [ ]:
# ---- adjust path to your snapshot ----
WEIGHTS = os.path.join(
    REPO_DIR,
    'output',
    'geotransformer.facesdownsampledfixed.stage4.gse.k3.max.oacl.stage2.sinkhorn.tripletratio',
    'snapshots',
    'epoch-40.pth.tar'
)


cfg = make_cfg()
model: GeoTransformer = create_model(cfg).to(device)
state_dict = torch.load(WEIGHTS, map_location=device)
model.load_state_dict(state_dict['model'])
model.eval()
print('Model loaded.')

### Load a sample pair

In [ ]:
def load_point_cloud(file_path, flip_z, flip_y, flip_x):
    pcd = o3d.io.read_point_cloud(file_path)
    print(pcd)
    pcd_d = pcd # pcd.uniform_down_sample(2)
    points = np.asarray(pcd_d.points, dtype=np.float32)
    center = points.mean(axis=0)
    points -= center
    scale = points[:, 1].max() - points[:, 1].min()
    scaled_points = points * 1.5 / scale
    t_noise = np.random.uniform(-1, 1, size=scaled_points.shape[1]).astype(np.float32)
    scaled_center_t_points = scaled_points + t_noise
    if flip_z:
        scaled_center_t_points[:, 2] *= -1
    if flip_y:
        scaled_center_t_points[:, 1] *= -1
    if flip_x:  
        scaled_center_t_points[:, 0] *= -1
    return scaled_center_t_points
PLY_FILE = "2189.ply"

In [ ]:
SCALE_SRC = True

In [ ]:
np.random.seed(23)

In [ ]:
# ---- adjust to your data path ----
DEMO_DIR = os.path.join(REPO_DIR, 'data', 'faces', 'demo')
SAMPLE_IDX = 22

#src_points_np = np.load(os.path.join(DEMO_DIR, f'src_{SAMPLE_IDX}.npy')).astype(np.float32)
src_points_np = load_point_cloud(PLY_FILE, flip_z=True, flip_y=True, flip_x=False)
ref_points_np = np.load(os.path.join(DEMO_DIR, f'ref_{SAMPLE_IDX}.npy')).astype(np.float32)
transform_np  = np.load(os.path.join(DEMO_DIR, f'gt_{SAMPLE_IDX}.npy')).astype(np.float32)

if SCALE_SRC:
    scale_factor =  1 + np.random.uniform(0.3, 0.8)
    src_points_np *= scale_factor
    transform_np[:3, :3] /= scale_factor
    print(f'Scaling source by factor {scale_factor:.2f}')

# dummy gt_z — not needed at inference
data_dict_raw = {
    'ref_points':    ref_points_np,
    'src_points':    src_points_np,
    'ref_feats':     np.ones_like(ref_points_np[:, :1]),
    'src_feats':     np.ones_like(src_points_np[:, :1]),
    'transform':     transform_np,
    'morphed_full':  np.zeros_like(ref_points_np),   # not used at inference
    'gt_z':          np.zeros((32, 100), dtype=np.float32),
}

print(f'ref: {ref_points_np.shape}  src: {src_points_np.shape}')

---
## Step 1 — Data Preparation: Multi-scale KNN graphs

In [ ]:
NEIGHBOR_LIMITS = [38, 36, 36, 38]   # calibrated defaults

data_dict = registration_collate_fn_stack_mode(
    [data_dict_raw],
    cfg.backbone.num_stages,
    cfg.backbone.init_voxel_size,
    cfg.backbone.init_radius,
    NEIGHBOR_LIMITS,
)
data_dict = to_cuda(data_dict)

print('Multi-scale point counts per cloud (ref | src):')
for i, (pts, lengths) in enumerate(zip(data_dict['points'], data_dict['lengths'])):
    n_ref = lengths[0].item()
    n_src = lengths[1].item()
    total = pts.shape[0]
    print(f'  Scale {i}: total={total:5d}  ref={n_ref:5d}  src={n_src:5d}')

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

scales = data_dict['points']
lengths = data_dict['lengths']

fig = make_subplots(
    rows=1, cols=4,
    subplot_titles=[f'Scale {i}' for i in range(4)],
    specs=[[{'type': 'scatter3d'}] * 4]
)

for i in range(4):
    n_ref = lengths[i][0].item()
    pts = scales[i][:n_ref].cpu().numpy()
    fig.add_trace(
        go.Scatter3d(
            x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
            mode='markers',
            marker=dict(size=1, opacity=0.6),
            name=f'Scale {i} ({n_ref} pts)'
        ),
        row=1, col=i+1
    )

fig.update_layout(
    title='Reference point cloud at each backbone scale',
    height=500,
    showlegend=True
)
fig.show()


In [ ]:
fig = make_subplots(
    rows=1, cols=4,
    subplot_titles=[f'Scale {i}' for i in range(4)],
    specs=[[{'type': 'scatter3d'}] * 4]
)

for i in range(4):
    start_idx = lengths[i][0].item()
    pts = scales[i][start_idx:].cpu().numpy()
    n_points = lengths[i][1].item()
    fig.add_trace(
        go.Scatter3d(
            x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
            mode='markers',
            marker=dict(size=1, opacity=0.6),
            name=f'Scale {i} ({n_points} pts)'
        ),
        row=1, col=i+1
    )

fig.update_layout(
    title='Src point cloud at each backbone scale',
    height=500,
    showlegend=True
)
fig.show()

---
## Step 2 — Pass-1 KPConvFPN Backbone

Encodes **original** ref+src features through 4 encoder stages and 2 decoder (FPN) stages.
Returns a list `[fine_feats, medium_feats, coarse_feats]` (reversed after decoder).

In [ ]:
with torch.no_grad():
    feats_list_pass1 = model.backbone(data_dict['features'], data_dict)

scale_names = ['fine (s2)', 'medium (s3)', 'coarse (s4)']
for name, feats in zip(scale_names, feats_list_pass1):
    print(f'{name:15s}: shape={tuple(feats.shape)}  mean={feats.mean():.4f}  std={feats.std():.4f}')

In [ ]:
# Feature norms across scales
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name, feats in zip(axes, scale_names, feats_list_pass1):
    norms = feats.norm(dim=1).cpu().numpy()
    ax.hist(norms, bins=60, edgecolor='none')
    ax.set_title(f'Pass-1 feature norms — {name}')
    ax.set_xlabel('L2 norm')
    ax.set_ylabel('count')
plt.tight_layout()
plt.show()

---
## Step 3 — Shape Morphing: PCA Coefficient Regression

The coarse src features are fed through the `CrossAttentionRegressor`.
32 learnable patch tokens attend to the src cloud via cross-attention
and each predicts 100 PCA coefficients → `z_delta` of shape (32, 100).

In [ ]:
with torch.no_grad():
    # isolate src at coarse level
    ref_length_orig = data_dict['lengths'][0][0].item()
    orig_points     = data_dict['points'][0]
    src_points_step3 = orig_points[ref_length_orig:]
    src_feats_raw    = data_dict['features'][ref_length_orig:]

    coarse_feats_all = feats_list_pass1[-1]         # coarse = last
    ref_len_c        = data_dict['lengths'][-1][0].item()
    src_coarse_feats = coarse_feats_all[ref_len_c:]

    # run regressor
    src_feats_batched = src_coarse_feats.unsqueeze(0)
    padding_mask = torch.zeros((1, src_coarse_feats.shape[0]), dtype=torch.bool, device=device)
    z_delta_batched = model.coeff_regressor(src_feats_batched, padding_mask)
    z_delta = z_delta_batched.squeeze(0)   # (32, 100)

print('z_delta (PCA coefficients):')
print(f'  shape : {tuple(z_delta.shape)}  (32 patches × 100 PCA components)')
print(f'  mean  : {z_delta.mean():.4f}')
print(f'  std   : {z_delta.std():.4f}')
print(f'  range : [{z_delta.min():.4f}, {z_delta.max():.4f}]')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Heatmap of z_delta (patches × components)
im = axes[0].imshow(z_delta.cpu().numpy(), aspect='auto', cmap='RdBu_r')
axes[0].set_title('Predicted PCA coefficients z_delta')
axes[0].set_xlabel('PCA component')
axes[0].set_ylabel('Patch index')
plt.colorbar(im, ax=axes[0])

# Distribution of coefficient magnitudes
axes[1].hist(z_delta.cpu().numpy().ravel(), bins=60)
axes[1].set_title('Distribution of z_delta values')
axes[1].set_xlabel('coefficient value')
axes[1].set_ylabel('count')

plt.tight_layout()
plt.show()

---
## Step 4 — Generate Morphed Reference

`reconstruction = pca_mean + z_delta @ pca_basis`  
Patches are stitched together (last-write-wins for overlapping vertices).

In [ ]:
with torch.no_grad():
    morphed_ref = model.generate_reference_geometry(z_delta)   # (N_verts, 3)
    recon_gt    = model.generate_reference_geometry(data_dict['gt_z'].squeeze(0) if data_dict['gt_z'].dim()==3 else data_dict['gt_z'])

print('Morphed reference shape :', tuple(morphed_ref.shape))

In [ ]:
import plotly.graph_objects as go

def make_scatter3d(pts, color, name):
    if torch.is_tensor(pts):
        pts = pts.cpu().numpy()
    return go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        mode='markers',
        marker=dict(size=1, color=color, opacity=0.5),
        name=name
    )

traces = [
    (make_scatter3d(ref_points_np, 'steelblue', 'Original reference'), 1),
    (make_scatter3d(morphed_ref, 'crimson', 'Predicted morphed reference'), 2),
    (make_scatter3d(src_points_np, 'darkorange', 'Source (scan)'), 3),
]

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Original reference', 'Predicted morphed reference', 'Source (scan)'],
    specs=[[{'type': 'scatter3d'}] * 3]
)

for trace, col in traces:
    fig.add_trace(trace, row=1, col=col)

fig.update_layout(title='Step 4 — Shape morphing output', height=500)
fig.show()


In [ ]:
# Interactive Open3D comparison
morphed_np = morphed_ref.cpu().numpy()

pcd_orig = make_open3d_point_cloud(ref_points_np)
pcd_orig.paint_uniform_color([0.0, 0.47, 0.72])   # blue  = original ref

pcd_morphed = make_open3d_point_cloud(morphed_np)
pcd_morphed.paint_uniform_color([0.84, 0.19, 0.15])  # red   = morphed ref

pcd_src = make_open3d_point_cloud(src_points_np)
pcd_src.paint_uniform_color([1.0, 0.70, 0.0])    # yellow = src

print('Blue=original ref  |  Red=morphed ref  |  Yellow=src')
o3d.visualization.draw_plotly([pcd_orig, pcd_morphed, pcd_src])

---
## Step 5 — Pass-2 KPConvFPN on Morphed Ref + Src

The morphed reference replaces the original ref. The backbone is re-run so that features
for rigid alignment reflect the personalised reference geometry.
The morphed points are **detached** so PCA regression and rigid alignment losses are independent.

In [ ]:
with torch.no_grad():
    morphed_ref_det = morphed_ref.detach()

    new_points = torch.cat([morphed_ref_det, src_points_step3], dim=0)
    new_lengths = torch.tensor([len(morphed_ref_det), len(src_points_step3)], dtype=torch.int32, device=device)

    # patch the data_dict for pass-2
    data_dict['points'][0] = new_points
    data_dict['lengths'][0] = new_lengths

    orig_feat_dim   = data_dict['features'].shape[1]
    new_ref_feats   = torch.ones((len(morphed_ref_det), orig_feat_dim), device=device)
    data_dict['features'] = torch.cat([new_ref_feats, src_feats_raw], dim=0)

    feats_list_pass2 = model.backbone(data_dict['features'], data_dict)

print('Pass-2 feature shapes:')
for name, feats in zip(scale_names, feats_list_pass2):
    print(f'  {name:15s}: {tuple(feats.shape)}')

In [ ]:
# Compare pass-1 vs pass-2 coarse feature norms for src cloud
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, feats_p1, feats_p2, name in zip(
    axes,
    [feats_list_pass1[0], feats_list_pass1[-1]],
    [feats_list_pass2[0], feats_list_pass2[-1]],
    ['fine', 'coarse']
):
    ref_len = data_dict['lengths'][-1 if name=='coarse' else 1][0].item()
    n1 = feats_p1[ref_len:].norm(dim=1).cpu().numpy()
    n2 = feats_p2[ref_len:].norm(dim=1).cpu().numpy()
    min_len = min(len(n1), len(n2))
    ax.scatter(n1[:min_len], n2[:min_len], s=3, alpha=0.4)
    ax.plot([n1.min(), n1.max()], [n1.min(), n1.max()], 'r--', lw=1)
    ax.set_title(f'Src {name} feature norms: pass-1 vs pass-2')
    ax.set_xlabel('pass-1 norm'); ax.set_ylabel('pass-2 norm')

plt.tight_layout()
plt.show()

---
## Step 6 — Point-to-Node Partitioning

Fine-level points are assigned to the nearest coarse superpoint (node).
Each node owns a patch of up to `num_points_in_patch=64` fine neighbours.

In [ ]:
with torch.no_grad():
    feats_c = feats_list_pass2[-1]
    feats_f = feats_list_pass2[0]

    ref_length_c = data_dict['lengths'][-1][0].item()
    ref_length_f = data_dict['lengths'][1][0].item()
    ref_length   = data_dict['lengths'][0][0].item()

    points_c = data_dict['points'][-1].detach()
    points_f = data_dict['points'][1].detach()
    points   = data_dict['points'][0].detach()

    ref_points_c = points_c[:ref_length_c]
    src_points_c = points_c[ref_length_c:]
    ref_points_f = points_f[:ref_length_f]
    src_points_f = points_f[ref_length_f:]

    _, ref_node_masks, ref_node_knn_indices, ref_node_knn_masks = point_to_node_partition(
        ref_points_f, ref_points_c, model.num_points_in_patch
    )
    _, src_node_masks, src_node_knn_indices, src_node_knn_masks = point_to_node_partition(
        src_points_f, src_points_c, model.num_points_in_patch
    )

print(f'Ref superpoints : {ref_points_c.shape[0]}  (mask non-empty: {ref_node_masks.sum().item()})')
print(f'Src superpoints : {src_points_c.shape[0]}  (mask non-empty: {src_node_masks.sum().item()})')
print(f'knn_indices shape: {ref_node_knn_indices.shape}  (nodes × max_k)')

In [ ]:
# Visualise one patch: pick the central superpoint and show its fine neighbours
ref_pts_c_np = ref_points_c.cpu().numpy()
ref_pts_f_np = ref_points_f.cpu().numpy()

# Pick the node closest to centroid
centroid = ref_pts_c_np.mean(axis=0)
node_idx = np.linalg.norm(ref_pts_c_np - centroid, axis=1).argmin()

patch_indices = ref_node_knn_indices[node_idx].cpu().numpy()
patch_mask    = ref_node_knn_masks[node_idx].cpu().numpy().astype(bool)
patch_pts     = ref_pts_f_np[patch_indices[patch_mask]]

centre = ref_pts_c_np[node_idx]

fig = go.Figure([
    go.Scatter3d(
        x=ref_pts_c_np[:, 0], y=ref_pts_c_np[:, 1], z=ref_pts_c_np[:, 2],
        mode='markers', marker=dict(size=2, color='lightgray', opacity=0.3),
        name='all superpoints'
    ),
    go.Scatter3d(
        x=patch_pts[:, 0], y=patch_pts[:, 1], z=patch_pts[:, 2],
        mode='markers', marker=dict(size=3, color='steelblue'),
        name=f'patch fine pts ({len(patch_pts)})'
    ),
    go.Scatter3d(
        x=[centre[0]], y=[centre[1]], z=[centre[2]],
        mode='markers', marker=dict(size=8, color='red'),
        name='superpoint centre'
    ),
])

fig.update_layout(title=f'Step 6 — Superpoint #{node_idx} and its fine patch', height=600)
fig.show()


---
## Step 7 — Geometric Structure Embedding

For each pair of superpoints the model encodes:
- **Pairwise distance** scaled by `sigma_d`
- **Triplet-wise angles** w.r.t. the k=3 nearest neighbours of each point, reduced with `max`

Both are encoded via a sinusoidal embedding and summed into a relative position bias
used inside the transformer attention.

In [ ]:
with torch.no_grad():
    ref_embeddings = model.transformer.embedding(ref_points_c.unsqueeze(0))  # (1, N, N, hidden)
    src_embeddings = model.transformer.embedding(src_points_c.unsqueeze(0))  # (1, M, M, hidden)

print('ref_embeddings shape:', ref_embeddings.shape)
print('src_embeddings shape:', src_embeddings.shape)

In [ ]:
# Visualise pairwise distance and angle embedding components
with torch.no_grad():
    d_idx, a_idx = model.transformer.embedding.get_embedding_indices(ref_points_c.unsqueeze(0))

d_map = d_idx[0].cpu().numpy()   # (N, N)
a_map = a_idx[0, :, :, 0].cpu().numpy()   # (N, N) for first neighbour angle

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im0 = axes[0].imshow(d_map, cmap='viridis')
axes[0].set_title('Pairwise distance index (ref superpoints)')
axes[0].set_xlabel('node j'); axes[0].set_ylabel('node i')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(a_map, cmap='plasma')
axes[1].set_title('Triplet angle index (k=1, ref superpoints)')
axes[1].set_xlabel('node j'); axes[1].set_ylabel('node i')
plt.colorbar(im1, ax=axes[1])

plt.suptitle('Step 7 — Geometric Structure Embedding indices')
plt.tight_layout()
plt.show()

---
## Step 8 — Geometric Transformer

Alternating self-attention (within one cloud) and cross-attention (between clouds),
conditioned on the geometric embeddings from Step 7.
Block sequence: `[self, cross, self, cross, self, cross]`.

In [ ]:
with torch.no_grad():
    ref_feats_c_raw = feats_c[:ref_length_c]
    src_feats_c_raw = feats_c[ref_length_c:]

    ref_feats_c, src_feats_c = model.transformer(
        ref_points_c.unsqueeze(0),
        src_points_c.unsqueeze(0),
        ref_feats_c_raw.unsqueeze(0),
        src_feats_c_raw.unsqueeze(0),
    )
    ref_feats_c = ref_feats_c.squeeze(0)
    src_feats_c = src_feats_c.squeeze(0)

    ref_feats_c_norm = F.normalize(ref_feats_c, p=2, dim=1)
    src_feats_c_norm = F.normalize(src_feats_c, p=2, dim=1)

print('Coarse features after GeometricTransformer:')
print(f'  ref: {ref_feats_c_norm.shape}  mean_norm={ref_feats_c_norm.norm(dim=1).mean():.3f}')
print(f'  src: {src_feats_c_norm.shape}  mean_norm={src_feats_c_norm.norm(dim=1).mean():.3f}')

In [ ]:
# Cross-similarity matrix between ref and src coarse features
with torch.no_grad():
    sim_matrix = torch.mm(ref_feats_c_norm, src_feats_c_norm.T).cpu().numpy()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(sim_matrix, cmap='hot', aspect='auto')
ax.set_title('Step 8 — Cosine similarity: ref × src coarse features')
ax.set_xlabel('src superpoint'); ax.set_ylabel('ref superpoint')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# Compare feature norms before vs after transformer
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, raw, tfm, label in zip(
    axes,
    [ref_feats_c_raw, src_feats_c_raw],
    [ref_feats_c, src_feats_c],
    ['ref', 'src']
):
    ax.hist(raw.norm(dim=1).cpu().numpy(), bins=40, alpha=0.6, label='before transformer')
    ax.hist(tfm.norm(dim=1).cpu().numpy(), bins=40, alpha=0.6, label='after transformer')
    ax.set_title(f'{label} coarse feature norms')
    ax.legend()
plt.tight_layout()
plt.show()

---
## Step 9 — Coarse Matching (SuperPoint Matching)

Computes pairwise feature distances between all ref and src superpoints,
applies dual normalisation, and selects the top-256 pairs.

In [ ]:
with torch.no_grad():
    ref_node_corr_indices, src_node_corr_indices, node_corr_scores = model.coarse_matching(
        ref_feats_c_norm, src_feats_c_norm, ref_node_masks, src_node_masks
    )

print(f'Coarse correspondences : {len(ref_node_corr_indices)}')
print(f'Score range            : [{node_corr_scores.min():.4f}, {node_corr_scores.max():.4f}]')

In [ ]:
# Show the matched superpoints in 3D
ref_matched = ref_points_c[ref_node_corr_indices].cpu().numpy()
src_matched = src_points_c[src_node_corr_indices].cpu().numpy()
scores_np   = node_corr_scores.cpu().numpy()

ref_donwsampled = scales[-2][:lengths[-2][0].item()].cpu().numpy()
src_downsampled = scales[-2][lengths[-2][0].item():].cpu().numpy()


# Keep top-30 for clarity
TOP = 10
top_idx = np.argsort(scores_np)[::-1][:TOP]

src_pts_c_np = src_points_c.cpu().numpy()

# Build correspondence lines (NaN-separated for a single trace)
line_x, line_y, line_z = [], [], []
for i in top_idx:
    r, s = ref_matched[i], src_matched[i]
    line_x += [r[0], s[0], None]
    line_y += [r[1], s[1], None]
    line_z += [r[2], s[2], None]

fig = go.Figure([
    go.Scatter3d(
        x=ref_pts_c_np[:, 0], y=ref_pts_c_np[:, 1], z=ref_pts_c_np[:, 2],
        mode='markers', marker=dict(size=2, color='steelblue', opacity=0.3),
        name='ref superpoints'
    ),
    go.Scatter3d(
        x=ref_donwsampled[:, 0], y=ref_donwsampled[:, 1], z=ref_donwsampled[:, 2],
        mode='markers', marker=dict(size=1, color='cyan', opacity=0.2),
        name='ref downsampled (for reference)'
    ),
    go.Scatter3d(
        x=src_pts_c_np[:, 0], y=src_pts_c_np[:, 1], z=src_pts_c_np[:, 2],
        mode='markers', marker=dict(size=2, color='darkorange', opacity=0.8),
        name='src superpoints'
    ),
    go.Scatter3d(
        x=src_downsampled[:, 0], y=src_downsampled[:, 1], z=src_downsampled[:, 2],
        mode='markers', marker=dict(size=1, color='green', opacity=0.3),
        name='src downsampled (for reference)'
    ),
    
    go.Scatter3d(
        x=line_x, y=line_y, z=line_z,
        mode='lines', line=dict(color='green', width=1),
        opacity=0.6, name=f'top-{TOP} matches'
    ),
    
])

fig.update_layout(title=f'Step 9 — Top-{TOP} coarse superpoint matches', height=600)
fig.show()


In [ ]:
# Score distribution
plt.figure(figsize=(7, 4))
plt.hist(scores_np, bins=50)
plt.title('Step 9 — Coarse correspondence score distribution')
plt.xlabel('score')
plt.ylabel('count')
plt.tight_layout()
plt.show()

---
## Step 10 — Fine Matching Preparation

For each coarse match, gather the fine-level kNN point patches from both clouds.
This results in tensors of shape `(num_coarse_matches, max_patch_size, 3/C)`.

In [ ]:
with torch.no_grad():
    ref_padded_points_f = torch.cat([ref_points_f, torch.zeros_like(ref_points_f[:1])], dim=0)
    src_padded_points_f = torch.cat([src_points_f, torch.zeros_like(src_points_f[:1])], dim=0)
    ref_node_knn_points = index_select(ref_padded_points_f, ref_node_knn_indices, dim=0)
    src_node_knn_points = index_select(src_padded_points_f, src_node_knn_indices, dim=0)

    ref_node_corr_knn_indices = ref_node_knn_indices[ref_node_corr_indices]
    src_node_corr_knn_indices = src_node_knn_indices[src_node_corr_indices]
    ref_node_corr_knn_masks   = ref_node_knn_masks[ref_node_corr_indices]
    src_node_corr_knn_masks   = src_node_knn_masks[src_node_corr_indices]
    ref_node_corr_knn_points  = ref_node_knn_points[ref_node_corr_indices]
    src_node_corr_knn_points  = src_node_knn_points[src_node_corr_indices]

    feats_f = feats_list_pass2[0]
    ref_feats_f = feats_f[:ref_length_f]
    src_feats_f = feats_f[ref_length_f:]
    ref_padded_feats_f = torch.cat([ref_feats_f, torch.zeros_like(ref_feats_f[:1])], dim=0)
    src_padded_feats_f = torch.cat([src_feats_f, torch.zeros_like(src_feats_f[:1])], dim=0)
    ref_node_corr_knn_feats = index_select(ref_padded_feats_f, ref_node_corr_knn_indices, dim=0)
    src_node_corr_knn_feats = index_select(src_padded_feats_f, src_node_corr_knn_indices, dim=0)

print('Fine patch tensors:')
print(f'  ref_node_corr_knn_points : {tuple(ref_node_corr_knn_points.shape)}')
print(f'  src_node_corr_knn_points : {tuple(src_node_corr_knn_points.shape)}')
print(f'  ref_node_corr_knn_feats  : {tuple(ref_node_corr_knn_feats.shape)}')

In [ ]:
# Show a single patch pair
PAIR_IDX = 0
r_mask = ref_node_corr_knn_masks[PAIR_IDX].cpu().numpy().astype(bool)
s_mask = src_node_corr_knn_masks[PAIR_IDX].cpu().numpy().astype(bool)
r_pts  = ref_node_corr_knn_points[PAIR_IDX][r_mask].cpu().numpy()
s_pts  = src_node_corr_knn_points[PAIR_IDX][s_mask].cpu().numpy()

fig = go.Figure([
    go.Scatter3d(
        x=r_pts[:, 0], y=r_pts[:, 1], z=r_pts[:, 2],
        mode='markers', marker=dict(size=3, color='steelblue'),
        name=f'ref patch ({len(r_pts)} pts)'
    ),
    go.Scatter3d(
        x=s_pts[:, 0], y=s_pts[:, 1], z=s_pts[:, 2],
        mode='markers', marker=dict(size=3, color='darkorange'),
        name=f'src patch ({len(s_pts)} pts)'
    ),
])

fig.update_layout(title=f'Step 10 — Fine patch pair #{PAIR_IDX}', height=600)
fig.show()


---
## Step 11 — Optimal Transport (Sinkhorn)

For each patch pair a dot-product similarity matrix is computed between fine features.
The learnable log-Sinkhorn algorithm iteratively normalises rows and columns
to produce a soft assignment (transport plan).

In [ ]:
with torch.no_grad():
    matching_scores_raw = torch.einsum(
        'bnd,bmd->bnm',
        ref_node_corr_knn_feats,
        src_node_corr_knn_feats
    ) / feats_f.shape[1] ** 0.5

    matching_scores = model.optimal_transport(
        matching_scores_raw,
        ref_node_corr_knn_masks,
        src_node_corr_knn_masks
    )

print(f'Sinkhorn output shape: {tuple(matching_scores.shape)}')
print(f'  (num_pairs={matching_scores.shape[0]}, ref_patch+1={matching_scores.shape[1]}, src_patch+1={matching_scores.shape[2]})')

In [ ]:
# Show 4 Sinkhorn assignment matrices (excluding dustbin row/col)
n_show = min(4, matching_scores.shape[0])
fig, axes = plt.subplots(1, n_show, figsize=(5 * n_show, 5))
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    mat = matching_scores[i, :-1, :-1].exp().cpu().numpy()   # log -> prob, drop dustbin
    r_m = ref_node_corr_knn_masks[i].cpu().numpy()
    s_m = src_node_corr_knn_masks[i].cpu().numpy()
    mat_valid = mat[:r_m.sum(), :s_m.sum()]
    im = ax.imshow(mat_valid, cmap='hot', aspect='auto', vmin=0)
    ax.set_title(f'Pair {i}\n({r_m.sum()}×{s_m.sum()})')
    ax.set_xlabel('src fine pts')
    ax.set_ylabel('ref fine pts')
    plt.colorbar(im, ax=ax)

plt.suptitle('Step 11 — Sinkhorn soft assignment matrices (exp of log scores)')
plt.tight_layout()
plt.show()

---
## Step 12 — Local-Global Registration (Fine Matching)

1. For each patch pair: pick top-k fine correspondences from Sinkhorn scores.
2. Weighted Procrustes gives a local rigid transform per patch.
3. Local transforms are aggregated and iteratively refined into a single global transform.

In [ ]:
with torch.no_grad():
    ms = matching_scores
    if not model.fine_matching.use_dustbin:
        ms = ms[:, :-1, :-1]

    ref_corr_points, src_corr_points, corr_scores, estimated_transform = model.fine_matching(
        ref_node_corr_knn_points,
        src_node_corr_knn_points,
        ref_node_corr_knn_masks,
        src_node_corr_knn_masks,
        ms,
        node_corr_scores,
    )

print(f'Fine correspondences  : {len(ref_corr_points)}')
print(f'Correspondence scores : [{corr_scores.min():.4f}, {corr_scores.max():.4f}]')
print(f'\nEstimated transform (4×4):')
print(estimated_transform.cpu().numpy())

In [ ]:
# Decompose estimated_transform into rotation, translation, and scale
T = estimated_transform.cpu().numpy()   # (4, 4)

M = T[:3, :3]   # upper-left 3×3 encodes s*R

# Uniform scale: average of column norms (columns of s*R each have magnitude s)
col_norms = np.linalg.norm(M, axis=0)   # (3,)
scale = col_norms.mean()

# Rotation matrix (remove scale)
R = M / col_norms[np.newaxis, :]        # divide each column by its norm

# Translation
translation = T[:3, 3]

# Verify R is orthogonal
ortho_err = np.linalg.norm(R @ R.T - np.eye(3))

print("=== Decomposition of estimated_transform ===")
print(f"\nScale      : {scale:.6f}  (column norms: {col_norms})")
print(f"\nRotation R :\n{R}")
print(f"\nTranslation: {translation}")
print(f"\nOrthogonality error of R (||R R^T - I||): {ortho_err:.2e}")

In [ ]:
# Visualise fine correspondences
r_corr_np = ref_corr_points.cpu().numpy()
s_corr_np = src_corr_points.cpu().numpy()
cs_np     = corr_scores.cpu().numpy()

# Subsample for readability
VIS_N = min(20, len(r_corr_np))
idx_vis = np.random.choice(len(r_corr_np), VIS_N, replace=False)

# Build correspondence lines with per-segment colour via score-mapped opacity
line_x, line_y, line_z = [], [], []
for i in idx_vis:
    line_x += [r_corr_np[i, 0], s_corr_np[i, 0], None]
    line_y += [r_corr_np[i, 1], s_corr_np[i, 1], None]
    line_z += [r_corr_np[i, 2], s_corr_np[i, 2], None]

fig = go.Figure([
    go.Scatter3d(
        x=r_corr_np[idx_vis, 0], y=r_corr_np[idx_vis, 1], z=r_corr_np[idx_vis, 2],
        mode='markers', marker=dict(size=2, color='steelblue'),
        name='ref corr pts'
    ),
    go.Scatter3d(
        x=s_corr_np[idx_vis, 0], y=s_corr_np[idx_vis, 1], z=s_corr_np[idx_vis, 2],
        mode='markers', marker=dict(size=2, color='darkorange'),
        name='src corr pts'
    ),
    go.Scatter3d(
        x=ref_donwsampled[:, 0], y=ref_donwsampled[:, 1], z=ref_donwsampled[:, 2],
        mode='markers', marker=dict(size=2, color='cyan', opacity=0.1),
        name='ref downsampled (for reference)'
    ),
    go.Scatter3d(
        x=src_downsampled[:, 0], y=src_downsampled[:, 1], z=src_downsampled[:, 2],
        mode='markers', marker=dict(size=2, color='green', opacity=0.1),
        name='src downsampled (for reference)'
    ),
    go.Scatter3d(
        x=line_x, y=line_y, z=line_z,
        mode='lines', line=dict(color='green', width=1),
        opacity=0.5, name='correspondences'
    ),
])

fig.update_layout(title=f'Step 12 — Fine correspondences (showing {VIS_N}/{len(r_corr_np)})', height=600)
fig.show()


In [ ]:

T_est = estimated_transform.cpu().numpy()
T_gt  = data_dict['transform'].cpu().numpy()

# Apply transform
src_fine_h   = np.hstack([s_corr_np, np.ones((len(s_corr_np), 1))])
src_fine_est = (T_est @ src_fine_h.T).T[:, :3]
src_fine_gt  = (T_gt  @ src_fine_h.T).T[:, :3]

src_fine_h_selected = src_fine_h[idx_vis]
src_fine_est_selected = src_fine_est[idx_vis]
src_fine_gt_selected  = src_fine_gt[idx_vis]

rre, rte = compute_registration_error(T_gt, T_est)
print(f'RRE (deg) : {rre:.3f}')
print(f'RTE (m)   : {rte:.4f}')

In [ ]:
def pts_trace(pts, color, name, opacity=0.5):
    if torch.is_tensor(pts):
        pts = pts.cpu().numpy()
    return go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        mode='markers', marker=dict(size=2, color=color, opacity=opacity),
        name=name
    )

panels = [
    {
        'traces': [
            pts_trace(ref_donwsampled, 'cyan', 'reference (original)', opacity=0.2),
            pts_trace(r_corr_np[idx_vis], 'steelblue', 'reference (morphed)'),
            pts_trace(src_fine_h_selected, 'darkorange', 'source'),
        ],
        'title': 'Before registration'
    },
    {
        'traces': [
            pts_trace(ref_donwsampled, 'cyan', 'reference (original)', opacity=0.2),
            pts_trace(r_corr_np[idx_vis], 'steelblue', 'reference (morphed)'),
            pts_trace(src_fine_est_selected, 'crimson', 'source (estimated)'),
        ],
        'title': f'After estimated transform<br>RRE={rre:.2f}° RTE={rte:.4f}m'
    },
    {
        'traces': [
            pts_trace(ref_donwsampled, 'cyan', 'reference (original)', opacity=0.2),
            pts_trace(r_corr_np[idx_vis], 'steelblue', 'reference (morphed)'),
            pts_trace(src_fine_gt_selected, 'seagreen', 'source (GT)'),
        ],
        'title': 'After GT transform'
    },
]

fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=[p['title'] for p in panels],
    specs=[[{'type': 'scatter3d'}]] * 3
)

for row, panel in enumerate(panels, start=1):
    for trace in panel['traces']:
        fig.add_trace(trace, row=row, col=1)

fig.update_layout(title='Step 12 — Matching result', height=2100, width=700)
fig.show()


---
## Step 13 — Registration Result

Apply the estimated transform to src and compare with the ground-truth transform.

In [ ]:
ref_pts_final = points[:ref_length].cpu().numpy()
src_pts_final = points[ref_length:].cpu().numpy()
T_est = estimated_transform.cpu().numpy()
T_gt  = data_dict['transform'].cpu().numpy()

# Apply transform
src_h   = np.hstack([src_pts_final, np.ones((len(src_pts_final), 1))])
src_est = (T_est @ src_h.T).T[:, :3]
src_gt  = (T_gt  @ src_h.T).T[:, :3]

rre, rte = compute_registration_error(T_gt, T_est)
print(f'RRE (deg) : {rre:.3f}')
print(f'RTE (m)   : {rte:.4f}')

In [ ]:
def pts_trace(pts, color, name):
    if torch.is_tensor(pts):
        pts = pts.cpu().numpy()
    return go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        mode='markers', marker=dict(size=1, color=color, opacity=0.5),
        name=name
    )

panels = [
    {
        'traces': [
            pts_trace(ref_pts_final, 'steelblue', 'reference (morphed)'),
            pts_trace(src_pts_final, 'darkorange', 'source'),
        ],
        'title': 'Before registration'
    },
    {
        'traces': [
            pts_trace(ref_pts_final, 'steelblue', 'reference (morphed)'),
            pts_trace(src_est, 'crimson', 'source (estimated)'),
        ],
        'title': f'After estimated transform<br>RRE={rre:.2f}° RTE={rte:.4f}m'
    },
    {
        'traces': [
            pts_trace(ref_pts_final, 'steelblue', 'reference (morphed)'),
            pts_trace(src_gt, 'seagreen', 'source (GT)'),
        ],
        'title': 'After GT transform'
    },
]

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[p['title'] for p in panels],
    specs=[[{'type': 'scatter3d'}] * 3]
)

for col, panel in enumerate(panels, start=1):
    for trace in panel['traces']:
        fig.add_trace(trace, row=1, col=col)

fig.update_layout(title='Step 13 — Registration result', height=500)
fig.show()


In [ ]:
# Interactive Open3D — before / after
pcd_ref = make_open3d_point_cloud(ref_pts_final)
pcd_ref.paint_uniform_color([0.0, 0.47, 0.72])   # blue  = ref

pcd_src_before = make_open3d_point_cloud(src_pts_final)
pcd_src_before.paint_uniform_color([1.0, 0.70, 0.0])  # yellow = src before

pcd_src_after = make_open3d_point_cloud(src_est)
pcd_src_after.paint_uniform_color([0.84, 0.19, 0.15])  # red = src after

print('Blue=ref  |  Yellow=src before  |  Red=src after estimated transform')
o3d.visualization.draw_plotly([pcd_ref, pcd_src_before, pcd_src_after])

---
## Summary

| Step | Output tensor | Shape |
|------|---------------|-------|
| 1 — Data prep | multi-scale point lists | 4 scales |
| 2 — Pass-1 backbone | `feats_list_pass1` | 3 scales × (N+M, C) |
| 3 — PCA coeff regression | `z_delta` | (32, 100) |
| 4 — Morphed reference | `morphed_ref` | (N_verts, 3) |
| 5 — Pass-2 backbone | `feats_list_pass2` | 3 scales × (N+M, C) |
| 6 — Partitioning | `ref/src_node_knn_indices` | (nodes, max_k) |
| 7 — Geometric embeddings | `ref/src_embeddings` | (1, nodes, nodes, H) |
| 8 — GeometricTransformer | `ref/src_feats_c_norm` | (nodes, 256) |
| 9 — Coarse matching | corr indices + scores | (256,) |
| 10 — Fine prep | `*_knn_feats/points` | (256, 64, C/3) |
| 11 — Sinkhorn | `matching_scores` | (256, 65, 65) |
| 12 — LGR | `estimated_transform` | (4, 4) |
